# DeltaGraph — graph queries on a 1 TB Databricks warehouse, no ETL

*"Customers who bought this also bought…"* — expressed as **one Cypher query**,
translated to Spark SQL and executed **in place** on Databricks' own TPC-DS benchmark
(`samples.tpcds_sf1000`, ~1 TB, ships in every workspace).

No graph database. Nothing copied. The warehouse does the work.

> **Before recording:** run the setup cell once **~30 s ahead of time**. The free-tier
> SQL warehouse auto-suspends when idle, so the *first* query pays a ~15–30 s cold start.
> The setup cell absorbs that here, off camera, so the demo cells run on the warm ~1 s path.

## Setup — connect to DeltaGraph

In [1]:
import os, time, requests, pandas as pd

# DeltaGraph HTTP endpoint. It speaks to a live Databricks SQL Warehouse;
# nothing runs locally except the Cypher→SQL translation.
DELTAGRAPH_HTTP = os.environ.get("DELTAGRAPH_HTTP", "http://localhost:7477/query")

def cypher_df(query):
    """Run a Cypher query on DeltaGraph → (DataFrame, seconds). Executes on Databricks."""
    t = time.perf_counter()
    r = requests.post(DELTAGRAPH_HTTP, json={"query": query}, timeout=300)
    r.raise_for_status()
    secs = time.perf_counter() - t
    return pd.DataFrame(r.json().get("results", [])), secs

def cypher_sql(query):
    """The Spark SQL DeltaGraph generates for a Cypher query (translation only, no run)."""
    r = requests.post(DELTAGRAPH_HTTP, json={"query": query, "sql_only": True}, timeout=60)
    r.raise_for_status()
    return r.json()["generated_sql"]

# Warm the warehouse (cold start absorbed HERE, not in the demo cells below)
_df, _s = cypher_df("RETURN 1 AS ok")
print(f"Connected to DeltaGraph. Warehouse warm-up: {_s:.1f}s")

Connected to DeltaGraph. Warehouse warm-up: 1.2s


## 1 · The scale

`store_sales` is the purchase edge: `(Customer)-[:PURCHASED]->(Item)`.
How many purchase edges are we about to traverse?

In [2]:
edges, secs = cypher_df(
    "MATCH (:Customer)-[r:PURCHASED]->(:Item) RETURN count(r) AS purchase_edges"
)
n = int(edges['purchase_edges'][0])
print(f"{n:,} purchase edges  (~{n/1e9:.2f} billion)  —  counted in {secs:.1f}s")

2,750,383,088 purchase edges  (~2.75 billion)  —  counted in 1.0s


## 2 · The graph question

*"Customers who bought item 13 also bought…"* — a bounded 2-hop co-purchase traversal
with a ranked aggregation. One Cypher query:

```cypher
MATCH (i1:Item {item_sk: 13})<-[:PURCHASED]-(c:Customer)-[:PURCHASED]->(i2:Item)
WHERE i2.item_sk <> 13 AND i2.name IS NOT NULL
RETURN i2.name AS also_bought, count(DISTINCT c) AS shoppers
ORDER BY shoppers DESC LIMIT 10
```

In [3]:
copurchase = """
MATCH (i1:Item {item_sk: 13})<-[:PURCHASED]-(c:Customer)-[:PURCHASED]->(i2:Item)
WHERE i2.item_sk <> 13 AND i2.name IS NOT NULL
RETURN i2.name AS also_bought, count(DISTINCT c) AS shoppers
ORDER BY shoppers DESC LIMIT 10
"""

df, secs = cypher_df(copurchase)
print(f"Self-join over ~{n/1e9:.2f}B purchase edges → 10 ranked rows in {secs:.2f}s")
df

Self-join over ~2.75B purchase edges → 10 ranked rows in 0.98s


,also_bought,shoppers
0,oughtesen stationn stought,14618
1,oughtbareseesecallyought,13132
2,antioughtpriableeing,10222
3,pribarableoughtableable,8799
4,bareingationoughtbarought,8016
5,oughtableablebaroughtought,7312
6,ableableprieingeseought,7124
7,eingeingesen stationable,7072
8,callycallyeingableoughtable,6516
9,esecallyoughtpricallyable,6484


> *TPC-DS product names are synthetic ("priought", "oughtesen stationn stought") —
> this is the real benchmark data, generator artifacts and all. The **ranking** and the
> **shopper counts** are what the co-purchase graph is actually computing.*

## 3 · No black box — the exact SQL it generated

`sql_only` returns the Spark SQL DeltaGraph produced. The 2-hop co-purchase lowers to a
plain FK-edge **self-join** on `store_sales` — nothing you'd want to hand-write, and
nothing hidden.

In [4]:
print(cypher_sql(copurchase))

SELECT 
      i2.i_product_name AS `also_bought`, 
      count(DISTINCT c.c_customer_sk) AS `shoppers`
FROM samples.tpcds_sf1000.store_sales AS t1
INNER JOIN samples.tpcds_sf1000.customer AS c ON t1.ss_customer_sk = c.c_customer_sk
INNER JOIN samples.tpcds_sf1000.store_sales AS t2 ON t2.ss_customer_sk = c.c_customer_sk
INNER JOIN samples.tpcds_sf1000.item AS i2 ON i2.i_item_sk = t2.ss_item_sk
WHERE (((i2.i_item_sk <> 13 AND i2.i_product_name IS NOT NULL) AND t1.ss_item_sk = 13) AND (t2.ss_customer_sk <> t1.ss_customer_sk OR t2.ss_item_sk <> t1.ss_item_sk))
GROUP BY i2.i_product_name
ORDER BY shoppers DESC NULLS FIRST
LIMIT 10


## One Cypher query. A terabyte. ~1 second. Zero data moved.

The same Cypher runs on **ClickHouse** too — point the endpoint at a ClickGraph server
and the identical query executes there. One graph language, any warehouse.

**github.com/genezhang/clickgraph**